# Analyzing the Impact of Economic Recession on Automobile Sales

**IBM / Coursera — Final Assignment Part 1: Matplotlib, Seaborn & Folium**

This independently reconstructed notebook follows the nine tasks described in the course assignment. It uses the **official IBM Skills Network data**; results are not simulated. The course starter notebook was not supplied, so this is not represented as an edited original.

## Data source and setup

Download the original dataset once with `python scripts/fetch_dataset.py` from the repository root. The notebook refuses to substitute synthetic data. Set the working directory to the repository root or the notebook directory.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import folium

sns.set_theme(style="whitegrid")
DATA_PATH = next((p for p in [Path("../data/historical_automobile_sales.csv"), Path("data/historical_automobile_sales.csv")] if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Official IBM dataset missing. Run `python scripts/fetch_dataset.py` from repository root.")
df = pd.read_csv(DATA_PATH)
required = {"Year", "Month", "Automobile_Sales", "Vehicle_Type", "Recession", "GDP", "Price", "Seasonality_Weight", "Advertising_Expenditure", "unemployment_rate"}
assert required <= set(df), f"Missing columns: {sorted(required - set(df))}"
assert df["Recession"].isin([0,1]).all()
df.head()

### Dataset quality checks

Report row count, date range, recession classes and missing values before interpreting charts.

In [ ]:
print(f"Rows: {len(df):,}; years: {df['Year'].min()}–{df['Year'].max()}")
print("Recession flags:", df['Recession'].value_counts().to_dict())
print("Nulls:", df.isna().sum().to_dict())

## Task 1.1 — Yearly automobile sales over time

A yearly mean is used to compare performance across years with consistent monthly granularity.

In [ ]:
annual = df.groupby('Year', as_index=False)['Automobile_Sales'].mean()
fig, ax = plt.subplots(figsize=(13,5))
ax.plot(annual['Year'], annual['Automobile_Sales'], marker='o', markersize=3)
ax.set(title='Average Automobile Sales by Year', xlabel='Year', ylabel='Average Automobile Sales')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()

## Task 1.2 — Advertising expenditure and sales in non-recession periods

Two separate scales avoid implying that advertising spending and sales have identical units.

In [ ]:
non_rec = df.loc[df['Recession'] == 0]
non_rec_yearly = non_rec.groupby('Year', as_index=False)[['Advertising_Expenditure','Automobile_Sales']].mean()
fig, ax1 = plt.subplots(figsize=(13,5))
line1 = ax1.plot(non_rec_yearly['Year'], non_rec_yearly['Advertising_Expenditure'], color='tab:blue', label='Advertising expenditure')
ax2 = ax1.twinx()
line2 = ax2.plot(non_rec_yearly['Year'], non_rec_yearly['Automobile_Sales'], color='tab:orange', label='Automobile sales')
ax1.set(xlabel='Year', ylabel='Mean advertising expenditure', title='Advertising Expenditure and Automobile Sales — Non-Recession Years')
ax2.set_ylabel('Mean automobile sales')
ax1.legend(line1+line2, [l.get_label() for l in line1+line2], loc='upper left')
plt.tight_layout(); plt.show()

## Task 1.3 — Grouped vehicle-type bar chart

Exact Seaborn syntax required by the assignment. Bars show mean sales, not total sales.

In [ ]:
fig, ax = plt.subplots(figsize=(11,5))
sns.barplot(x='Vehicle_Type', y='Automobile_Sales', hue='Recession', data=df, ax=ax)
ax.set(title='Average Automobile Sales by Vehicle Type and Recession Status', xlabel='Vehicle Type', ylabel='Average Automobile Sales')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout(); plt.show()

## Task 1.4 — GDP subplots

Compare recession and non-recession GDP observations separately, preserving their observed time positions.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,5), sharey=True)
for ax, flag, title in zip(axes, [1,0], ['Recession Periods','Non-Recession Periods']):
    t = df.loc[df['Recession'] == flag].groupby('Year', as_index=False)['GDP'].mean()
    ax.plot(t['Year'], t['GDP'], marker='o', markersize=3)
    ax.set(title=f'Average GDP — {title}', xlabel='Year', ylabel='Average GDP')
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()

## Task 1.5 — Seasonality bubble chart

Bubble area represents automobile sales while axes show month and seasonality weight.

In [ ]:
season = df.groupby('Month',as_index=False).agg(Seasonality_Weight=('Seasonality_Weight','mean'),Automobile_Sales=('Automobile_Sales','mean'))
size = 1200 * season['Automobile_Sales'] / season['Automobile_Sales'].max()
fig, ax = plt.subplots(figsize=(11,5))
sc = ax.scatter(season['Month'],season['Seasonality_Weight'],s=size,alpha=.55,c=season['Automobile_Sales'],cmap='viridis',edgecolor='white')
fig.colorbar(sc,ax=ax,label='Average automobile sales')
ax.set(title='Monthly Seasonality Weight and Automobile Sales',xlabel='Month',ylabel='Average seasonality weight',xticks=range(1,13))
plt.tight_layout(); plt.show()

## Task 1.6 — Price and automobile sales during recessions

The scatter plot shows association only; it does not establish causation.

In [ ]:
rec = df.loc[df['Recession'] == 1]
fig, ax = plt.subplots(figsize=(10,5))
ax.scatter(rec['Price'],rec['Automobile_Sales'],alpha=.55)
ax.set(title='Average Vehicle Price vs Automobile Sales — Recession Periods',xlabel='Average Vehicle Price',ylabel='Automobile Sales')
plt.tight_layout(); plt.show()

## Task 1.7 — Share of advertising expenditure by economic period

Sum spending over recorded observations in each recession category.

In [ ]:
ad_period = df.groupby('Recession')['Advertising_Expenditure'].sum().reindex([0,1])
fig, ax = plt.subplots(figsize=(7,7))
ax.pie(ad_period,labels=['Non-recession','Recession'],autopct='%1.1f%%',startangle=90)
ax.set_title('Advertising Expenditure Share: Recession vs Non-Recession')
plt.tight_layout(); plt.show()

## Task 1.8 — Advertising allocation by vehicle type during recession

In [ ]:
ad_vehicle = rec.groupby('Vehicle_Type')['Advertising_Expenditure'].sum().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8,8))
ax.pie(ad_vehicle,labels=ad_vehicle.index,autopct='%1.1f%%',startangle=90)
ax.set_title('Advertising Expenditure by Vehicle Type During Recessions')
plt.tight_layout(); plt.show()

## Task 1.9 — Unemployment rate and automobile sales by vehicle type

Aggregate observations with identical unemployment rates and vehicle types to show a line for each category.

In [ ]:
unemployment_sales = rec.groupby(['Vehicle_Type','unemployment_rate'],as_index=False)['Automobile_Sales'].mean()
fig, ax = plt.subplots(figsize=(11,6))
sns.lineplot(data=unemployment_sales,x='unemployment_rate',y='Automobile_Sales',hue='Vehicle_Type',marker='o',ax=ax)
ax.set(title='Unemployment Rate and Automobile Sales by Vehicle Type — Recessions',xlabel='Unemployment Rate',ylabel='Average Automobile Sales')
plt.tight_layout(); plt.show()

## Folium note

The nine specified tasks above do not include a geographic map. The IBM dataset must have legitimate coordinates or a documented region-to-coordinate mapping before a Folium map can be generated. Folium is imported as requested; no fictional geolocation is created.

## Interpretation and reproducibility

Only make numerical claims after actually executing this notebook on the official dataset. The project README and `docs/validation_report.md` explicitly describe the current validation status.

## Data-grounded findings

The following results are computed directly from the official IBM records when executed. They describe association, not causation.

In [ ]:
comparison = df.groupby('Recession')['Automobile_Sales'].agg(['count','mean','median']).rename(index={0:'Non-recession', 1:'Recession'})
vehicle_comparison = df.groupby(['Vehicle_Type', 'Recession'])['Automobile_Sales'].mean().unstack()
print('Sales by period:')
print(comparison.to_string())
print('\nMean sales by vehicle type and period:')
print(vehicle_comparison.to_string())
print('\nMean difference, recession minus non-recession:', round(float(comparison.loc['Recession','mean'] - comparison.loc['Non-recession','mean']),2))